In [ ]:
import seaborn as sns
import pandas as pd
import matplotlib.pyplot as plt
import json

def plot_results(filepath_list):
    data_list = []
    for filepath in filepath_list:
        with open(filepath) as file:
            data_list.extend(json.load(file))

    data = data_list
    df = pd.DataFrame(data)

    # Select relevant numerical metrics for plotting
    metrics_to_plot = [
        "env/charging_stops_per_episode_mean",
        "env/global_ttt",
        "env/global_ttt_only_terminated",
        "env/ttt_per_ev_mean",
        "env/ttt_per_ev_mean_only_terminated",
        "env/cumulated_waiting_time",
        "env/cumulated_waiting_time_only_terminated",
        "env/empty_vehicles_per_episode",
        "env/final_simulation_time",
    ]

    # Melt the DataFrame to long format
    df_melted = df.melt(id_vars=["algorithm"], 
                        value_vars=metrics_to_plot, 
                        var_name="Metric", 
                        value_name="Value")

    # Create plots
    sns.set_theme(style="whitegrid")

    for metric in metrics_to_plot:
        plt.figure(figsize=(10, 6))

        # Filter data for the current metric
        df_filtered = df_melted[df_melted["Metric"] == metric]

        sns.violinplot(x="algorithm", y="Value", data=df_filtered, palette="muted", cut = 0)
        # add in for smaller datasets:
        # sns.stripplot(df_filtered, x="algorithm", y="Value", color=".3")

        plt.title(f"Comparison of {metric}", fontsize=14)
        plt.xlabel("Algorithm")
        plt.ylabel("Value")
        plt.grid(axis="y", linestyle="-", alpha=0.7) 

        sns.despine(left=True, bottom=True)
        plt.show()




In [ ]:
# Plot ratio of truncated episodes

import json
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# List of file paths
filepath_list = [
    "models/2025-03-23_15-16-01_v0.7.5_shaping_circle_RANDOM/evaluation/metrics2025-03-23_15-16-00.json",
    "models/2025-03-23_15-18-25_v0.7.5_shaping_circle_GREEDY/evaluation/metrics2025-03-23_15-18-24.json",
    "models/2025-03-10_11-47-34_v0.7.5_shaping_circle_A2C/evaluation/metrics2025-03-23_15-16-43.json",
]

# Read all JSON files and combine the data into one list
data_list = []
for filepath in filepath_list:
    with open(filepath, 'r') as file:
        data_list.extend(json.load(file))

# Convert the combined data to a DataFrame
df = pd.DataFrame(data_list)

# Group by algorithm and calculate:
# - total episodes (by counting rows)
# - number of truncated episodes (summing the boolean column)
summary = df.groupby('algorithm').agg(
    total_episodes=('episode', 'count'),
    truncated_episodes=('was_truncated', 'sum')
).reset_index()

# Calculate the truncated ratio for each algorithm
summary['truncated_ratio'] = summary['truncated_episodes'] / summary['total_episodes']

print(summary)

# Plot the truncated ratio for each algorithm using seaborn
plt.figure(figsize=(8, 6))
sns.barplot(data=summary, x='algorithm', y='truncated_ratio')
plt.title("Fraction of Truncated Episodes per Algorithm")
plt.xlabel("Algorithm")
plt.ylabel("Fraction of Episodes Truncated")
plt.ylim(0, 1)  # Ratio between 0 and 1
plt.show()


In [ ]:
df[df["algorithm"] == "A2C"].head()

In [ ]:
# plot episode length

summary_ep_length = df.groupby('algorithm').agg(
    episode_length_mean=('episode_length', 'mean'),
    simulation_length_mean=('env/final_simulation_time', 'mean')
).reset_index()

summary_ep_length

In [ ]:
compare_filepaths_list = [
    # "models/2025-03-28_17-19-31_v0.7.5_shaping_circle_RANDOM/evaluation/metrics2025-03-28_17-19-30.json",
    "models/2025-04-04_11-47-36_v0.7.5_basic_circleSameSOC_GREEDY/evaluation/metrics2025-04-04_11-47-35.json",
    # "models/2025-03-28_16-56-14_v0.7.5_basic_circleSameSOC_A2C/evaluation/metrics2025-03-28_17-29-42.json",
    "models/2025-04-04_10-51-41_v0.7.5_basic_circleSameSOC_PPO/evaluation/metrics2025-04-04_11-54-28.json"

]
plot_results(compare_filepaths_list)